In [ ]:
!pip install -q transformers sentencepiece torch accelerate PyPDF2 python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.0 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files
from PyPDF2 import PdfReader
from docx import Document
import textwrap
import torch
import os

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on: cpu


In [ ]:
def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_pdf(file_path):
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

def read_docx(file_path):
    doc = Document(file_path)
    text = []
    for para in doc.paragraphs:
        if para.text.strip():
            text.append(para.text)
    return "\n".join(text)

def extract_text_from_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".txt":
        return read_txt(file_path)
    elif ext == ".pdf":
        return read_pdf(file_path)
    elif ext == ".docx":
        return read_docx(file_path)
    elif ext == ".doc":
        raise ValueError("Legacy .doc files are not reliably supported in Colab. Please convert to .docx or PDF.")
    else:
        raise ValueError(f"Unsupported file type: {ext}")

In [ ]:
def split_text(text, max_words=180):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunks.append(" ".join(words[i:i+max_words]))
    return chunks

In [ ]:
def generate_summary(prompt, max_output_tokens=120):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_output_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def summarize_book(book_text):
    chunks = split_text(book_text, max_words=180)
    chunk_summaries = []

    for i, chunk in enumerate(chunks, start=1):
        print(f"Summarizing chunk {i}/{len(chunks)}...")
        prompt = f"Summarize the following content in a short and clear paragraph:\n\n{chunk}"
        summary = generate_summary(prompt, max_output_tokens=100)
        chunk_summaries.append(summary)

    combined = " ".join(chunk_summaries)

    final_prompt = f"Create a final overall summary from these summaries:\n\n{combined}"
    final_summary = generate_summary(final_prompt, max_output_tokens=120)

    return chunk_summaries, final_summary

In [ ]:
uploaded = files.upload()

Saving ICS LCA Reporttttt.pdf to ICS LCA Reporttttt.pdf


In [ ]:
uploaded_file_name = list(uploaded.keys())[0]
print("Uploaded file:", uploaded_file_name)

book_text = extract_text_from_file(uploaded_file_name)

print("\nExtracted text preview:\n")
print(book_text[:1000])

Uploaded file: ICS LCA Reporttttt.pdf

Extracted text preview:

 
Laboratory  Report  Network  & Security  
 
 
 
 
 
 
 
 
Journal  Report  
Network Security 
Information  and Cyber  Security  
 
 
Under  the Guidance  of 
Dr. Umesh Raut 
umesh.raut@mitwpu.edu.in  
 
 
Submitted  by: 
Divyateja Reddy Anakala 
1032220427@mitwpu.edu.in  
Devanshi Lambodari  
1032222453@mitwpu.edu.in  
Raj Patil  
1032222253@mitwpu.edu.in  
March  25, 2026  
 
Laboratory  Report  Network  & Security  
 
1   
Contents  
1 Introduction ............................................. 1  
2  Lab 1: Wireshark – Packet Sniffing, Protocol Analysis, and Detecting Suspicious Traffic 
............ 2  
  2.1 Objectives ........................................... 2  
  2.2 Procedure ............................................ 2  
  2.3 Results .............................................. 3  
  2.4 Conclusion ........................................... 3  
3 Lab 2: Tcpdump – CLI-Based Packet Capture and Filtering Ex

In [ ]:
chunk_summaries, final_output = summarize_book(book_text)

print("\n--- CHUNK SUMMARIES ---")
for i, s in enumerate(chunk_summaries, start=1):
    print(f"\nChunk {i} Summary:")
    print(textwrap.fill(s, width=100))

print("\n--- FINAL SUMMARY ---")
print(textwrap.fill(final_output, width=100))

Summarizing chunk 1/9...
Summarizing chunk 2/9...
Summarizing chunk 3/9...
Summarizing chunk 4/9...
Summarizing chunk 5/9...
Summarizing chunk 6/9...
Summarizing chunk 7/9...
Summarizing chunk 8/9...
Summarizing chunk 9/9...

--- CHUNK SUMMARIES ---

Chunk 1 Summary:
The Journal Report Network Security Information and Cyber Security Under the Guidance of Dr. Umesh
Raut (M.R.N.S.) is a peer-reviewed scientific journal published by the Indian Institute of
Technology, Bombay. It is the only academic journal in India to publish a scientific journal. It is
the only academic journal in India to publish a scientific journal. It is the only academic journal
in India to publish a scientific journal.

Chunk 2 Summary:
The Laboratory Report Network & Security 1 Chapter 1 Introduction This report presents practical
experiments conducted in the field of Network Security and Cyber Security . The objective of these
labs is to understand how different tools help in analyzing network traffic, detecting